# 02) Data alignment: fill VOD + auxvars into LFMC 0.1° grid-day table

目标：
- 读取 `lfmc_grid_day_0p1.parquet`（0.1° 像元-日 LFMC）
- 对每一行，根据 `lat/lon` → (row, col) 索引，从 1800×3600 影像数组中取值
- 将 **VOD（多频双极化）** 与 **辅助变量（CH/IGBP/AGB/LAI/FVC/LST）** 作为新列填充
- 导出基础训练数据表（parquet）

> 说明：MATLAB 写出的 HDF5 有时会出现维度顺序与 Python 读取不同（(1800,3600) vs (3600,1800)）。
本 notebook 会做 shape 检查，必要时自动转置，确保 `(row, col)` 与经纬度一致。

In [1]:
import os, sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import h5py
from datetime import datetime, timedelta

import importlib

# notebook 能 import function/ 下的模块（兼容 VSCode Jupyter 的 cwd 行为）
cwd = Path.cwd().resolve()
if (cwd / "configs" / "lfmc.yaml").exists() and (cwd / "function").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "configs" / "lfmc.yaml").exists() and (cwd.parent / "function").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("找不到项目根目录：未发现 configs/lfmc.yaml 与 function/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------- 关键：让 Python 发现更新后的 .py ----------
importlib.invalidate_caches()

def import_or_reload(module_name: str):
    """
    Import a module; if already imported, reload it so changes in .py take effect.
    """
    if module_name in sys.modules:
        return importlib.reload(sys.modules[module_name])
    return importlib.import_module(module_name)

# ---------- VOD 读取 + QC ----------
m_vod_h5 = import_or_reload("function.vod.vod_h5")
find_vod_file = m_vod_h5.find_vod_file
read_vod_h5 = m_vod_h5.read_vod_h5

m_vod_qc = import_or_reload("function.vod.vod_qc")
build_vod_qc_array = m_vod_qc.build_vod_qc_array
build_valid_mask = m_vod_qc.build_valid_mask
apply_qc_mask_to_vod = m_vod_qc.apply_qc_mask_to_vod

# ---------- auxvars loaders ----------
m_canopy = import_or_reload("function.auxvars.canopy_height")
load_canopy_height = m_canopy.load_canopy_height

m_agb = import_or_reload("function.auxvars.agb")
load_agb_static_2015 = m_agb.load_agb_static_2015

m_igbp = import_or_reload("function.auxvars.igbp")
load_igbp_year = m_igbp.load_igbp_year

m_lai = import_or_reload("function.auxvars.glass_lai")
load_lai_by_index = m_lai.load_lai_by_index

m_fvc = import_or_reload("function.auxvars.glass_fvc")
load_fvc_by_index = m_fvc.load_fvc_by_index

m_lst = import_or_reload("function.auxvars.lst")
load_lst_daily = m_lst.load_lst_daily

print("Modules imported/reloaded successfully.")


Modules imported/reloaded successfully.


In [2]:
def load_yaml(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

cfg_paths = load_yaml(PROJECT_ROOT / "configs" / "paths.yaml")
cfg_lfmc  = load_yaml(PROJECT_ROOT / "configs" / "lfmc.yaml")
cfg_aux   = load_yaml(PROJECT_ROOT / "configs" / "auxvars.yaml")

if "paths" in cfg_paths: cfg_paths = cfg_paths["paths"]
if "lfmc"  in cfg_lfmc:  cfg_lfmc  = cfg_lfmc["lfmc"]
if "auxvars" in cfg_aux: cfg_aux = cfg_aux["auxvars"]

print("Loaded configs:")
print(" - paths.yaml keys:", list(cfg_paths.keys()))
print(" - lfmc.yaml keys:", list(cfg_lfmc.keys()))
print(" - auxvars.yaml keys:", list(cfg_aux.keys()))

Loaded configs:
 - paths.yaml keys: ['vod', 'qc']
 - lfmc.yaml keys: ['raw_path', 'sheet_name', 'out_dir', 'out_std_name', 'out_dedup_name', 'out_report_name', 'out_grid_day_name', 'normalize_to_date', 'dedup_strategy', 'columns', 'lfmc_min', 'lfmc_max', 'suspect_keywords']
 - auxvars.yaml keys: ['grid', 'canopy_height', 'agb', 'igbp', 'lai', 'fvc', 'lst']


## 1) 读取 0.1° 像元-日 LFMC 表

`lfmc_grid_day_0p1.parquet` 的默认命名是 `lfmc_grid_day_0p1.parquet`，也可以在 `lfmc.yaml` 用 `out_grid_day_name` 指定。

In [3]:
out_dir = Path(cfg_lfmc["out_dir"])
grid_name = cfg_lfmc.get("out_grid_day_name", "lfmc_grid_day_0p1.parquet")
grid_path = out_dir / grid_name
if not grid_path.exists():
    raise FileNotFoundError(f"找不到 {grid_path}，请确认已从 01b_lfmc_read_clean.ipynb 导出。")

df_grid = pd.read_parquet(grid_path)
print("df_grid shape:", df_grid.shape)
print("columns:", list(df_grid.columns))
df_grid.head(3)

df_grid shape: (140112, 9)
columns: ['date', 'grid_lat', 'grid_lon', 'veg_type', 'lfmc_pct', 'n_obs_sum', 'n_sites', 'source', 'n_records']


,date,grid_lat,grid_lon,veg_type,lfmc_pct,n_obs_sum,n_sites,source,n_records
0,1977-04-14,37.25,-122.15,Shrub,176.0,1,1,US National Fuel Moisture Database https://www...,1
1,1977-04-14,37.45,-122.35,Shrub,203.0,1,1,US National Fuel Moisture Database https://www...,1
2,1977-04-15,37.05,-121.85,Shrub,96.0,1,1,US National Fuel Moisture Database https://www...,1


## 2) 将 lat/lon 转为 1800×3600 的行列索引

网格中心：
- 纬度：89.95, 89.85, ..., -89.95（步长 -0.1）
- 经度：-179.95, -179.85, ..., 179.95（步长 +0.1）

索引定义（与 MATLAB 的行/列方向一致）：
- `row = round((89.95 - lat) / 0.1)`
- `col = round((lon + 179.95) / 0.1)`

In [4]:
# grid geometry (0.1° global grid centers)
GRID_ROWS = int(cfg_aux["grid"]["rows"])   # 1800
GRID_COLS = int(cfg_aux["grid"]["cols"])   # 3600
LAT0 = float(cfg_aux["grid"]["lat0"])      # 89.95
LON0 = float(cfg_aux["grid"]["lon0"])      # -179.95
RES  = float(cfg_aux["grid"]["res"])       # 0.1


def latlon_to_rowcol(lat: np.ndarray, lon: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    lat/lon are grid CENTER coordinates.
    row increases southward, col increases eastward.
    """
    # 关键：round 到 2 位小数，消除 0.1 网格中心的浮点尾巴
    lat = np.round(lat.astype(np.float64), 2)
    lon = np.round(lon.astype(np.float64), 2)

    row = np.rint((LAT0 - lat) / RES).astype(np.int32)
    col = np.rint((lon - LON0) / RES).astype(np.int32)
    return row, col


def rowcol_to_latlon(row: np.ndarray, col: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    lat = np.round(LAT0 - row.astype(np.float64) * RES, 2)
    lon = np.round(LON0 + col.astype(np.float64) * RES, 2)
    return lat, lon


# ---------- choose correct column names ----------
lat_col = "grid_lat" if "grid_lat" in df_grid.columns else ("lat" if "lat" in df_grid.columns else None)
lon_col = "grid_lon" if "grid_lon" in df_grid.columns else ("lon" if "lon" in df_grid.columns else None)

if lat_col is None or lon_col is None or "date" not in df_grid.columns:
    raise KeyError(f"df_grid 需要包含 date + (grid_lat/grid_lon 或 lat/lon). 当前列: {list(df_grid.columns)}")

# ---------- compute row/col ----------
rows, cols = latlon_to_rowcol(df_grid[lat_col].to_numpy(), df_grid[lon_col].to_numpy())

# bounds check
in_bounds = (rows >= 0) & (rows < GRID_ROWS) & (cols >= 0) & (cols < GRID_COLS)
if not np.all(in_bounds):
    bad = np.where(~in_bounds)[0][:10]
    raise ValueError(
        "发现越界 row/col（展示前10条索引）: "
        f"{bad.tolist()}\n"
        f"示例 lat/lon: {df_grid.iloc[bad][[lat_col, lon_col]].to_dict('records')}"
    )

df_grid["row"] = rows
df_grid["col"] = cols

# ---------- optional: force “clean” grid center coords ----------
# 这一步可以直接把 grid_lat/grid_lon “纠正”为严格的 0.1°中心值（推荐）
lat_center, lon_center = rowcol_to_latlon(rows, cols)
df_grid["grid_lat"] = lat_center
df_grid["grid_lon"] = lon_center

# sanity check print
print("lat/lon columns used:", lat_col, lon_col)
print("sample centers:", list(zip(df_grid["grid_lat"].head(5).tolist(), df_grid["grid_lon"].head(5).tolist())))


lat/lon columns used: grid_lat grid_lon
sample centers: [(37.25, -122.15), (37.45, -122.35), (37.05, -121.85), (37.05, -121.85), (37.45, -122.35)]


## 3) HDF5 读取与 shape 纠正（MATLAB ↔ Python）

如果读到的数组是 `(3600,1800)`，会自动转置成 `(1800,3600)`，以保证：
- `arr[row, col]` 对应 `lat(row), lon(col)`

In [5]:
def ensure_shape_1800x3600(arr: np.ndarray, name: str = "") -> np.ndarray:
    if arr.shape == (GRID_ROWS, GRID_COLS):
        return arr
    if arr.shape == (GRID_COLS, GRID_ROWS):
        return arr.T
    raise ValueError(f"{name} unexpected shape: {arr.shape}, expect {(GRID_ROWS, GRID_COLS)} or {(GRID_COLS, GRID_ROWS)}")

def extract_at(arr: np.ndarray, row: np.ndarray, col: np.ndarray) -> np.ndarray:
    # arr must be (rows, cols)
    return arr[row, col]

## 4) 预加载静态/年尺度变量（CH, AGB, IGBP）

- CH：静态，全时段同一个文件
- AGB：这里按你的设定只取 2015 文件并视为静态
- IGBP：年度变量，按 `date.year` 选择 `YYYY001.h5`

In [6]:
# -------- static vars --------
print("Loading static canopy height...")
ch = load_canopy_height(cfg_aux["canopy_height"]["path"], cfg_aux["canopy_height"]["var"])
ch = ensure_shape_1800x3600(ch, "canopy_height")

print("Loading static AGB (2015)...")
agb = load_agb_static_2015(cfg_aux["agb"]["dir"], cfg_aux["agb"]["var"])
agb = ensure_shape_1800x3600(agb, "agb")

# cache for annual igbp
IGBP_VARS = cfg_aux["igbp"]["vars"]
igbp_cache: dict[int, dict[str, np.ndarray]] = {}

def get_igbp_year(year: int) -> dict[str, np.ndarray]:
    if year in igbp_cache:
        return igbp_cache[year]

    # 按 igbp.py 的真实签名调用
    d = load_igbp_year(
        igbp_dir=cfg_aux["igbp"]["dir"],
        year=year,
        classes=IGBP_VARS,
    )

    out = {k: ensure_shape_1800x3600(v, f"igbp_{k}_{year}") for k, v in d.items()}
    igbp_cache[year] = out
    return out

print("Static + annual loaders ready.")

Loading static canopy height...
Loading static AGB (2015)...
Static + annual loaders ready.


## 5) 定义 8-day 索引规则（LAI / FVC）

你的规则：  
> 例如 2002-01-02 位于 2002-01-01 ~ 2002-01-08，则取文件 20020101；  
> 下一期从 20020109 开始（2002-01-09 ~ 2002-01-16）。

实现：对任意日期，计算当年的 `doy`，索引起始日为 `((doy-1)//8)*8 + 1`。

In [7]:
def index_date_8day(d: pd.Timestamp) -> pd.Timestamp:
    d = pd.Timestamp(d).normalize()
    year = d.year
    doy = int(d.dayofyear)
    start_doy = ((doy - 1) // 8) * 8 + 1
    return pd.Timestamp(datetime(year, 1, 1) + timedelta(days=start_doy - 1))

# quick test
for x in ["2002-01-01", "2002-01-02", "2002-01-08", "2002-01-09", "2002-12-31"]:
    print(x, "->", index_date_8day(pd.Timestamp(x)).date())

2002-01-01 -> 2002-01-01
2002-01-02 -> 2002-01-01
2002-01-08 -> 2002-01-01
2002-01-09 -> 2002-01-09
2002-12-31 -> 2002-12-27


## 6) 主循环：按 date 分组读取影像并填充列

策略：
- 先对 `df_grid` 按日期分组
- 每个日期只读取一次当天（或当期）的影像数组，然后对该日期的所有行用 `(row,col)` 抽样
- 年度 IGBP 使用缓存（每年读一次）
- 静态变量 CH/AGB 只读一次

输出列：
- VOD: `tau_Ku_H, tau_Ku_V, tau_X_H, tau_X_V, tau_C_H, tau_C_V`
- aux: `canopy_height, agb, lai, fvc, lst`
- IGBP fractions: `igbp_ENF, ..., igbp_WAT`

In [8]:
# ---- prepare output columns ----
vod_var_map = cfg_paths["vod"]["var_map"]  # mat_var -> std_name
vod_std_names = list(vod_var_map.values())
qc_keep = cfg_paths["qc"]["keep_flags"]
qc_var = cfg_paths["vod"]["qc_var"]
vod_templates = cfg_paths["vod"]["filename_templates"]
vod_base = cfg_paths["vod"]["base_path"]

# initialize columns with NaN
for c in vod_std_names:
    df_grid[c] = np.nan

df_grid[cfg_aux["canopy_height"]["out_col"]] = extract_at(ch, df_grid["row"].to_numpy(), df_grid["col"].to_numpy()).astype(np.float32)
df_grid[cfg_aux["agb"]["out_col"]] = extract_at(agb, df_grid["row"].to_numpy(), df_grid["col"].to_numpy()).astype(np.float32)

# IGBP columns
igbp_prefix = cfg_aux["igbp"]["prefix"]
for v in IGBP_VARS:
    df_grid[f"{igbp_prefix}{v}"] = np.nan

# dynamic aux
df_grid[cfg_aux["lai"]["out_col"]] = np.nan
df_grid[cfg_aux["fvc"]["out_col"]] = np.nan
df_grid[cfg_aux["lst"]["out_col"]] = np.nan

# optional: store file index dates used for 8-day vars (debug/traceability)
df_grid["lai_index_date"] = pd.NaT
df_grid["fvc_index_date"] = pd.NaT

# ---- main loop ----
dates = pd.to_datetime(df_grid["date"]).dt.normalize()
df_grid["date"] = dates  # ensure normalized

groups = df_grid.groupby("date", sort=True)

print("Unique dates:", len(groups))

Unique dates: 8991


In [9]:
from tqdm import tqdm

for d, idx in tqdm(groups.indices.items(), total=len(groups), desc="Align per date"):
    # row indices in df_grid for this date
    sel = np.asarray(idx, dtype=np.int64)
    r = df_grid.loc[sel, "row"].to_numpy(dtype=np.int32)
    c = df_grid.loc[sel, "col"].to_numpy(dtype=np.int32)

    # ---------- VOD ----------
    try:
        vod_cfg = cfg_paths["vod"]
        qc_cfg  = cfg_paths["qc"]

        d_dt = pd.Timestamp(d).to_pydatetime()

        vod_file = find_vod_file(d_dt, vod_cfg)
        if vod_file is not None:
            # 1) read h5
            vod_dict, qc_uint16 = read_vod_h5(vod_file, vod_cfg)

            # 2) 强制 shape 校验（不符合直接报错，避免 silent）
            qc_uint16 = ensure_shape_1800x3600(qc_uint16, "vod_qc_raw").astype(np.uint16)
            for k in list(vod_dict.keys()):
                vod_dict[k] = ensure_shape_1800x3600(vod_dict[k], f"vod_{k}")

            # 3) 生成 vod_qc_flag (0~6) + 有效mask
            vod_qc_flag = build_vod_qc_array(
                qc_uint16=qc_uint16,
                vod_dict=vod_dict,
                fill_qc_value=int(qc_cfg.get("fill_qc_value", 255)),
            )
            valid_mask = build_valid_mask(
                vod_qc_array=vod_qc_flag,
                keep_flags=tuple(qc_cfg.get("keep_flags", [0])),
            )

            # 4) mask 后再抽样填充
            vod_masked = apply_qc_mask_to_vod(vod_dict, valid_mask, fill_value=np.nan)

            for std_name, arr in vod_masked.items():
                arr = ensure_shape_1800x3600(arr, f"vod_masked_{std_name}")
                df_grid.loc[sel, std_name] = extract_at(arr, r, c).astype(np.float32)

    except FileNotFoundError:
        pass

    # ---------- LST (daily) ----------
    try:
        # d 可能是 pandas Timestamp，这里统一转 python datetime
        d_dt = pd.Timestamp(d).to_pydatetime()

        lst_arr = load_lst_daily(
            cfg_aux["lst"]["dir"],
            d_dt,
            var_name=cfg_aux["lst"].get("var", "LST"),
        )
        lst_arr = ensure_shape_1800x3600(lst_arr, "lst")
        df_grid.loc[sel, cfg_aux["lst"]["out_col"]] = extract_at(lst_arr, r, c).astype(np.float32)
    except FileNotFoundError:
        pass

    # ---------- LAI/FVC (8-day index) ----------
    idx8 = index_date_8day(d)  # 你自己定义的规则：返回窗口起始日（Timestamp）
    idx8_dt = pd.Timestamp(idx8).to_pydatetime()

    try:
        lai_arr = load_lai_by_index(
            cfg_aux["lai"]["dir"],
            idx8_dt,
            var_name=cfg_aux["lai"].get("var", "LAI"),
        )
        lai_arr = ensure_shape_1800x3600(lai_arr, "lai")
        df_grid.loc[sel, cfg_aux["lai"]["out_col"]] = extract_at(lai_arr, r, c).astype(np.float32)
        df_grid.loc[sel, "lai_index_date"] = pd.Timestamp(idx8).normalize()
    except FileNotFoundError:
        pass

    try:
        fvc_arr = load_fvc_by_index(
            cfg_aux["fvc"]["dir"],
            idx8_dt,
            var_name=cfg_aux["fvc"].get("var", "FVC"),
        )
        fvc_arr = ensure_shape_1800x3600(fvc_arr, "fvc")
        df_grid.loc[sel, cfg_aux["fvc"]["out_col"]] = extract_at(fvc_arr, r, c).astype(np.float32)
        df_grid.loc[sel, "fvc_index_date"] = pd.Timestamp(idx8).normalize()
    except FileNotFoundError:
        pass

    # ---------- IGBP (annual cache) ----------
    target_year = int(pd.Timestamp(d).year)
    try:
        ig = get_igbp_year(target_year)
        for v in IGBP_VARS:
            df_grid.loc[sel, f"{igbp_prefix}{v}"] = extract_at(ig[v], r, c).astype(np.float32)
    except FileNotFoundError:
        pass

print("Done alignment.")

Align per date: 100%|██████████| 8991/8991 [2:39:16<00:00,  1.06s/it]  

Done alignment.


### 补：VOD文件出错的调试解决，检查文件是否有误，如果前置未处理，则强烈建议在执行6)之前能够检查一下所有的h5文件。代码放在matlab中进行修复。

In [10]:
# import os
# import re
# from typing import List, Dict, Any, Optional, Tuple
# import numpy as np
# import h5py
# import pandas as pd
# from concurrent.futures import ProcessPoolExecutor, as_completed

# # ----------------------
# # 文件收集
# # ----------------------
# def collect_all_h5_files(base_path: str) -> List[str]:
#     h5_files = []
#     for root, _, files in os.walk(base_path):
#         for fn in files:
#             if fn.lower().endswith(".h5"):
#                 h5_files.append(os.path.join(root, fn))
#     h5_files.sort()
#     return h5_files

# def parse_date_from_filename(path: str) -> Optional[str]:
#     m = re.search(r"(\d{8})", os.path.basename(path))
#     return m.group(1) if m else None

# # ----------------------
# # 单个dataset：抽样读chunk
# # ----------------------
# def _sample_chunk_starts(shape: Tuple[int, ...], chunks: Tuple[int, ...], k: int, seed: int) -> List[Tuple[int, ...]]:
#     """
#     返回 k 个随机 chunk 起点 + 3 个固定点（左上/中心/右下）。
#     """
#     rng = np.random.default_rng(seed)

#     # 每维 chunk 数
#     n_chunks = [max(1, int(np.ceil(shape[d] / chunks[d]))) for d in range(len(shape))]

#     starts = []

#     # 固定点：左上、中心、右下（按chunk格）
#     fixed = [
#         tuple(0 for _ in n_chunks),
#         tuple(nc // 2 for nc in n_chunks),
#         tuple(nc - 1 for nc in n_chunks),
#     ]
#     for idx in fixed:
#         start = tuple(int(idx[d] * chunks[d]) for d in range(len(shape)))
#         starts.append(start)

#     # 随机点（在chunk格上采样）
#     for _ in range(k):
#         idx = tuple(int(rng.integers(0, n_chunks[d])) for d in range(len(shape)))
#         start = tuple(int(idx[d] * chunks[d]) for d in range(len(shape)))
#         starts.append(start)

#     # 去重
#     starts = list(dict.fromkeys(starts))
#     return starts

# def _read_chunk(ds: h5py.Dataset, start: Tuple[int, ...], chunks: Tuple[int, ...]) -> None:
#     slicer = tuple(slice(start[d], min(start[d] + chunks[d], ds.shape[d])) for d in range(ds.ndim))
#     _ = ds[slicer]  # 触发解压

# # ----------------------
# # 单文件检查（在子进程里跑）
# # ----------------------
# def check_one_file_fast(args: Tuple[str, List[str], int, int, bool]) -> Optional[Dict[str, Any]]:
#     """
#     返回 None 表示没发现问题；否则返回坏文件记录（只返回第一个问题，追求速度）。
#     args:
#       - file_path
#       - dataset_paths_to_check (e.g., ['/x_vod_V','/x_vod_H',...])
#       - sample_chunks_per_ds
#       - seed_base
#       - do_full_read (慎用，慢)
#     """
#     file_path, ds_paths, sample_chunks_per_ds, seed_base, do_full_read = args
#     date_str = parse_date_from_filename(file_path)

#     try:
#         with h5py.File(file_path, "r") as f:
#             for dspath in ds_paths:
#                 if dspath not in f:
#                     return {
#                         "file": file_path,
#                         "date": date_str,
#                         "dataset": dspath,
#                         "issue": "missing_dataset",
#                         "error": "not found in file",
#                     }

#                 ds = f[dspath]
#                 meta = {
#                     "file": file_path,
#                     "date": date_str,
#                     "dataset": dspath,
#                     "shape": tuple(ds.shape),
#                     "dtype": str(ds.dtype),
#                     "chunks": tuple(ds.chunks) if ds.chunks is not None else None,
#                     "compression": ds.compression,
#                     "compression_opts": ds.compression_opts,
#                 }

#                 # 标量/非chunked：读小块即可
#                 if ds.ndim == 0:
#                     _ = ds[()]
#                     continue
#                 if ds.chunks is None:
#                     slicer = tuple(slice(0, min(8, s if s else 1)) for s in ds.shape)
#                     try:
#                         _ = ds[slicer]
#                     except Exception as e:
#                         return {**meta, "issue": "read_failure", "bad_chunk_start": None, "error": repr(e)}
#                     if do_full_read:
#                         try:
#                             _ = ds[:]
#                         except Exception as e:
#                             return {**meta, "issue": "full_read_failure", "bad_chunk_start": None, "error": repr(e)}
#                     continue

#                 # chunked：抽样读
#                 seed = (seed_base ^ hash((file_path, dspath))) & 0xFFFFFFFF
#                 starts = _sample_chunk_starts(ds.shape, ds.chunks, sample_chunks_per_ds, seed)

#                 for st in starts:
#                     try:
#                         _read_chunk(ds, st, ds.chunks)
#                     except Exception as e:
#                         return {**meta, "issue": "chunk_read_failure", "bad_chunk_start": st, "error": repr(e)}

#                 # 可选：全量读（非常慢，不建议全盘扫时开）
#                 if do_full_read:
#                     try:
#                         _ = ds[:]
#                     except Exception as e:
#                         return {**meta, "issue": "full_read_failure", "bad_chunk_start": None, "error": repr(e)}

#         return None

#     except Exception as e:
#         return {
#             "file": file_path,
#             "date": date_str,
#             "dataset": None,
#             "issue": "file_open_failure",
#             "error": repr(e),
#         }

# # ----------------------
# # 并行扫描主程序
# # ----------------------
# def scan_all_fast(vod_cfg: dict,
#                   max_workers: int = 4,
#                   sample_chunks_per_ds: int = 30,
#                   do_full_read: bool = False,
#                   seed_base: int = 12345,
#                   only_check_keys: Optional[List[str]] = None) -> pd.DataFrame:
#     base = vod_cfg["base_path"]
#     files = collect_all_h5_files(base)
#     print(f"[SCAN] {len(files)} h5 files under {base}")

#     # 要检查的dataset路径
#     if only_check_keys is None:
#         var_map = vod_cfg.get("var_map", {})
#         qc_var = vod_cfg.get("qc_var", "QC")
#         keys = list(var_map.keys()) + [qc_var]
#     else:
#         keys = only_check_keys

#     ds_paths = [k if str(k).startswith("/") else "/" + str(k) for k in keys]
#     print("[SCAN] datasets to check:", ds_paths)

#     tasks = [(fp, ds_paths, sample_chunks_per_ds, seed_base, do_full_read) for fp in files]
#     bad = []

#     # 经验：网络盘/机械盘 max_workers=2~6；本地SSD可更大
#     with ProcessPoolExecutor(max_workers=max_workers) as ex:
#         futs = [ex.submit(check_one_file_fast, t) for t in tasks]
#         for i, fut in enumerate(as_completed(futs), start=1):
#             res = fut.result()
#             if res is not None and res.get("issue") in ("chunk_read_failure", "full_read_failure", "file_open_failure", "missing_dataset", "read_failure"):
#                 bad.append(res)

#             if i % 200 == 0:
#                 print(f"[PROGRESS] done {i}/{len(files)}  bad_found={len(bad)}")

#     df = pd.DataFrame(bad)
#     print(f"[DONE] bad_found={len(df)}")
#     return df

# # ======================
# # 运行建议
# # ======================

# # 1) 最快粗筛：只查最可疑变量（例如你已发现 x_vod_V 会坏）
# # df_bad = scan_all_fast(cfg_paths["vod"], max_workers=4, sample_chunks_per_ds=30, only_check_keys=["x_vod_V"], do_full_read=False)

# # 2) 稍慢但更全面：查 var_map + QC（仍然是抽样chunk）
# df_bad = scan_all_fast(cfg_paths["vod"], max_workers=4, sample_chunks_per_ds=30, only_check_keys=None, do_full_read=False)

# # 保存坏文件清单
# # df_bad.to_csv("vod_bad_files_fastcheck.csv", index=False)

# df_bad.head(20)


## 7) 简单检查与导出

- 检查关键列缺失率
- 导出 `train_base_0p1.parquet`（可在 auxvars.yaml / lfmc.yaml 中自行重命名）

In [11]:
def missing_ratio(s: pd.Series) -> float:
    return float(s.isna().mean())

check_cols = [
    "lfmc_pct",
    *vod_std_names,
    cfg_aux["canopy_height"]["out_col"],
    cfg_aux["agb"]["out_col"],
    cfg_aux["lai"]["out_col"],
    cfg_aux["fvc"]["out_col"],
    cfg_aux["lst"]["out_col"],
]

print("Missing ratios:")
for c in check_cols:
    if c in df_grid.columns:
        print(f"  {c:20s} : {missing_ratio(df_grid[c]):.4f}")

# export
out_train = out_dir / cfg_lfmc.get("out_lfmc_base_name", "lfmc_base_0p1.parquet")

# 只导出需要的列（也可以把其它元数据一起保留）
keep_cols = [
    "date","lat","lon","row","col",
    "lfmc_pct","n_obs",
    "site_id","site_name","veg_type",
    "dead_protocol","qc_hard",
    *vod_std_names,
    cfg_aux["canopy_height"]["out_col"],
    cfg_aux["agb"]["out_col"],
    cfg_aux["lai"]["out_col"],
    cfg_aux["fvc"]["out_col"],
    cfg_aux["lst"]["out_col"],
    "lai_index_date","fvc_index_date",
    *[f"{igbp_prefix}{v}" for v in IGBP_VARS],
]
keep_cols = [c for c in keep_cols if c in df_grid.columns]

df_train = df_grid[keep_cols].copy()
df_train.to_parquet(out_train, index=False)
print("Saved:", out_train)
df_train.head(3)

Missing ratios:
  lfmc_pct             : 0.0000
  tau_Ku_H             : 0.6029
  tau_Ku_V             : 0.6831
  tau_X_H              : 0.5858
  tau_X_V              : 0.6202
  tau_C_H              : 0.5869
  tau_C_V              : 0.6753
  canopy_height        : 0.0000
  agb                  : 0.0000
  lai                  : 0.0878
  fvc                  : 0.0860
  lst                  : 0.0981
Saved: G:\data\Globe LFMC\processed\lfmc_base_0p1.parquet


,date,row,col,lfmc_pct,veg_type,tau_Ku_H,tau_Ku_V,tau_X_H,tau_X_V,tau_C_H,...,igbp_WSA,igbp_SAV,igbp_GRA,igbp_WET,igbp_CRO,igbp_URB,igbp_CVM,igbp_SNO,igbp_BAR,igbp_WAT
0,1977-04-14,527,578,176.0,Shrub,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1977-04-14,525,576,203.0,Shrub,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1977-04-15,529,581,96.0,Shrub,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8) 打印各个列名以及部分独立值

In [12]:
import pandas as pd
import numpy as np

parquet_path = r"G:\data\Globe LFMC\processed\lfmc_base_0p1.parquet"
df = pd.read_parquet(parquet_path)

print("列数:", len(df.columns))

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("\n列名列表（含数值列范围）:")
for i, c in enumerate(df.columns, 1):
    if c in num_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vmin = s.min(skipna=True)
        vmax = s.max(skipna=True)
        nn = int(s.notna().sum())
        print(f"{i:03d}. {c}  [numeric]  min={vmin}  max={vmax}  n={nn}")
    else:
        nn = int(df[c].notna().sum())
        print(f"{i:03d}. {c}  [non-numeric]  n={nn}")


列数: 35

列名列表（含数值列范围）:
001. date  [non-numeric]  n=140112
002. row  [numeric]  min=248  max=1328  n=140112
003. col  [numeric]  min=244  max=3309  n=140112
004. lfmc_pct  [numeric]  min=0.0  max=400.0  n=140112
005. veg_type  [non-numeric]  n=140112
006. tau_Ku_H  [numeric]  min=0.016449853777885437  max=4.710214614868164  n=55641
007. tau_Ku_V  [numeric]  min=0.0013648619642481208  max=5.446719169616699  n=44407
008. tau_X_H  [numeric]  min=0.021708691492676735  max=6.548359394073486  n=58033
009. tau_X_V  [numeric]  min=0.0012069708900526166  max=5.61576509475708  n=53211
010. tau_C_H  [numeric]  min=0.025132527574896812  max=3.023735284805298  n=57875
011. tau_C_V  [numeric]  min=0.00036028181784786284  max=5.0953521728515625  n=45488
012. canopy_height  [numeric]  min=0.0  max=39.52964401245117  n=140112
013. agb  [numeric]  min=0.0  max=377.2790832519531  n=140112
014. lai  [numeric]  min=0.0  max=5.775000095367432  n=127811
015. fvc  [numeric]  min=0.0  max=0.9620000123977661  n=1

In [13]:
import pandas as pd
import numpy as np

path = r"G:\data\Globe LFMC\processed\lfmc_base_0p1.parquet"
df = pd.read_parquet(path)

max_list = 200   # 你想最多打印多少个独立值（防爆）
result = {}

for col in df.columns:
    s = df[col]

    # 统计独立值数量（包含 NaN 与否可以选）
    nunique = s.nunique(dropna=False)

    # 判断是否“类别列”：object/category/bool，或 unique 数量不大
    is_categorical_like = (
        str(s.dtype) in ("object", "category", "bool") or nunique <= max_list
    )

    if is_categorical_like:
        # 列出全部（或最多 max_list 个）
        uniq_vals = s.drop_duplicates().head(max_list).tolist()
        truncated = nunique > max_list
    else:
        # 数值列：只取样一些独立值看一下
        uniq_vals = s.dropna().drop_duplicates().sample(
            n=min(30, max(1, s.dropna().nunique())), random_state=0
        ).tolist()
        truncated = True

    result[col] = {
        "dtype": str(s.dtype),
        "nunique(dropna=False)": int(nunique),
        "unique_values_preview": uniq_vals,
        "preview_truncated": bool(truncated),
    }

# 打印其中几列看看
for k, v in list(result.items())[:10]:
    print(k, v["dtype"], v["nunique(dropna=False)"])
    print(v["unique_values_preview"])
    print("truncated:", v["preview_truncated"])
    print("-"*60)


date datetime64[ns] 8991
[Timestamp('2005-02-22 00:00:00'), Timestamp('2008-05-04 00:00:00'), Timestamp('2003-01-22 00:00:00'), Timestamp('2008-03-14 00:00:00'), Timestamp('2020-04-17 00:00:00'), Timestamp('1998-06-24 00:00:00'), Timestamp('2020-09-03 00:00:00'), Timestamp('2016-12-04 00:00:00'), Timestamp('2016-10-07 00:00:00'), Timestamp('1990-07-06 00:00:00'), Timestamp('2009-06-02 00:00:00'), Timestamp('2013-07-11 00:00:00'), Timestamp('1989-11-17 00:00:00'), Timestamp('2019-02-15 00:00:00'), Timestamp('2013-07-03 00:00:00'), Timestamp('2011-08-22 00:00:00'), Timestamp('2015-01-27 00:00:00'), Timestamp('2013-04-09 00:00:00'), Timestamp('2012-05-07 00:00:00'), Timestamp('1984-08-25 00:00:00'), Timestamp('2000-09-24 00:00:00'), Timestamp('2005-10-21 00:00:00'), Timestamp('2014-08-29 00:00:00'), Timestamp('2022-01-27 00:00:00'), Timestamp('2012-07-04 00:00:00'), Timestamp('2003-06-04 00:00:00'), Timestamp('2021-05-16 00:00:00'), Timestamp('2017-02-01 00:00:00'), Timestamp('1997-07-19 

## 9）保存一版xlsx文件

In [15]:
import os
import pandas as pd

parquet_path = r"G:\data\Globe LFMC\processed\lfmc_base_0p1.parquet"
df = pd.read_parquet(parquet_path)

xlsx_path = os.path.splitext(parquet_path)[0] + ".xlsx"

# 为了“内容一致”，不改变列顺序、不改值；index=False 防止多一列索引
df.to_excel(xlsx_path, index=False, engine="openpyxl")

print("Saved:", xlsx_path)
print("Rows:", len(df), "Cols:", len(df.columns))


Saved: G:\data\Globe LFMC\processed\lfmc_base_0p1.xlsx
Rows: 140112 Cols: 35
